# Prompt Engineering Best Practices for Instruction-Tuned LLM [Part 2]

[Article](https://todatabeyond.com/blog/prompt-engineering-best-practices-for-instruction-tuned-llm-part-2). [Series contents](./README.md).

Install `pip install openai==2.43.0 python-dotenv==1.2.2`. Set OPENAI_API_KEY in your environment (or an uncommitted .env); API calls incur charges. MODEL defaults to the tested gpt-5.4-mini. Never place credentials in cells. The examples request ordinary task outputs; responses can change between runs. The nonexecuting appendix preserves original source code and transcripts.

In [ ]:
import os, json, re, sys
from importlib.metadata import version
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=60)
def get_completion(prompt, model=MODEL):
    result = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        reasoning_effort="none", max_completion_tokens=2048)
    if result.choices[0].finish_reason != "stop":
        raise RuntimeError("The completion did not finish normally")
    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("No text returned")
    return content
def parse_json_object(text):
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
    raise ValueError("No JSON object found")
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "python-dotenv:", version("python-dotenv"))
print("Model:", MODEL)

In [ ]:
text = f"""
In a charming village, siblings Jack and Jill set out on  a quest to fetch water from a hilltop  well. As they climbed, singing joyfully, misfortune  struck—Jack tripped on a stone and tumbled  down the hill, with Jill following suit.  Though slightly battered, the pair returned home to  comforting embraces. Despite the mishap,  their adventurous spirits remained undimmed, and they  continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple  backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following  keys: french_summary, num_names.

Separate your answers with line breaks.
Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)
first_steps_response = response

In [ ]:
prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by 
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>
Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)
formatted_steps_response = response

In [ ]:
prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need   help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost  me a flat $100k per year, and an additional $10 / square  foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completion(prompt)
print(response)
bare_math_response = response

In [ ]:
prompt = f"""
Your task is to determine if the student's solution  is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total. 
- Then compare your solution to the student's solution  and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until 
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution  just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help  working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost  me a flat $100k per year, and an additional $10 / square  foot
What is the total cost for the first year of operations  as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)
checked_math_response = response

In [ ]:
prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completion(prompt)
print(response)
product_response = response

In [ ]:
assert parse_json_object(first_steps_response)["num_names"] == 2
structured_steps = parse_json_object(formatted_steps_response)
assert structured_steps["num_names"] == 2
assert all(name in structured_steps["french_summary"] for name in ["Jack", "Jill"])
assert "360" in checked_math_response.replace(",", "")
assert "incorrect" in checked_math_response.lower()
def first_year_cost(area):
    if area < 0: raise ValueError("Area must be nonnegative")
    return (100 + 250 + 10) * area + 100_000
assert [first_year_cost(a) for a in [0, 1, 100]] == [100_000, 100_360, 136_000]
assert first_year_cost(100) != 450 * 100 + 100_000
assert len(product_response) > 20
print("PASS: both task-output JSON objects, two names, independent cost arithmetic and corrected student grade. Product response captured without assuming it is factual.")

## Original source appendix — not executed

Historical SDK calls and transcripts below are retained verbatim for comparison, not used as fresh test results.

### Original pre block 1

```text
text = f"""
In a charming village, siblings Jack and Jill set out on \ 
a quest to fetch water from a hilltop \ 
well. As they climbed, singing joyfully, misfortune \ 
struck—Jack tripped on a stone and tumbled \ 
down the hill, with Jill following suit. \ 
Though slightly battered, the pair returned home to \ 
comforting embraces. Despite the mishap, \ 
their adventurous spirits remained undimmed, and they \ 
continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple \
backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following \
keys: french_summary, num_names.

Separate your answers with line breaks.
Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)
```

### Original pre block 2

```text
prompt_2 = f"""
Your task is to perform the following actions: 
1 - Summarize the following text delimited by 
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the 
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>
Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)
```

### Original pre block 3

```text
prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need \
 help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \ 
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 4

```text
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total. 
- Then compare your solution to the student's solution \ 
and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until 
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 5

```text
prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completion(prompt)
print(response)
```

### Original quote/transcript 1

```text
Completion for prompt 1:
1 — Jack and Jill, siblings, go on a quest to fetch water from a hilltop well, but encounter misfortune when Jack trips on a stone and tumbles down the hill, with Jill following suit, yet they return home and remain undeterred in their adventurous spirits.
```

### Original quote/transcript 2

```text
2 — Jack et Jill, frère et sœur, partent en quête d’eau d’un puits au sommet d’une colline, mais rencontrent un malheur lorsque Jack trébuche sur une pierre et dévale la colline, suivi par Jill, pourtant ils rentrent chez eux et restent déterminés dans leur esprit d’aventure.
```

### Original quote/transcript 3

```text
3 — Jack, Jill
```

### Original quote/transcript 4

```text
4 — {
 “french_summary”: “Jack et Jill, frère et sœur, partent en quête d’eau d’un puits au sommet d’une colline, mais rencontrent un malheur lorsque Jack trébuche sur une pierre et dévale la colline, suivi par Jill, pourtant ils rentrent chez eux et restent déterminés dans leur esprit d’aventure.”,
 “num_names”: 2
}
```

### Original quote/transcript 5

```text
Completion for prompt 2:
Summary: Jack and Jill, siblings from a charming village, go on a quest to fetch water from a hilltop well, but encounter misfortune when Jack trips on a stone and tumbles down the hill, with Jill following suit, yet they remain undeterred and continue exploring with delight.
```

### Original quote/transcript 6

```text
Translation: Jack et Jill, frère et sœur d’un charmant village, partent à la recherche d’eau d’un puits au sommet d’une colline, mais rencontrent un malheur lorsque Jack trébuche sur une pierre et dévale la colline, suivi de Jill, pourtant ils restent déterminés et continuent à explorer avec joie.
```

### Original quote/transcript 7

```text
Names: Jack, Jill
```

### Original quote/transcript 8

```text
Output JSON: 
{
 “french_summary”: “Jack et Jill, frère et sœur d’un charmant village, partent à la recherche d’eau d’un puits au sommet d’une colline, mais rencontrent un malheur lorsque Jack trébuche sur une pierre et dévale la colline, suivi de Jill, pourtant ils restent déterminés et continuent à explorer avec joie.”,
 “num_names”: 2
}
```

### Original quote/transcript 9

```text
The student’s solution is correct. They correctly identified the costs for land, solar panels, and maintenance, and calculated the total cost as a function of the number of square feet.
```

### Original quote/transcript 10

```text
To calculate the total cost for the first year of operations, we need to add up the costs of land, solar panels, and maintenance.
```

### Original quote/transcript 11

```text
1. Land cost: $100 / square foot
The cost of land is $100 multiplied by the size of the installation in square feet.
```

### Original quote/transcript 12

```text
2. Solar panel cost: $250 / square foot
The cost of solar panels is $250 multiplied by the size of the installation in square feet.
```

### Original quote/transcript 13

```text
3. Maintenance cost: $100,000 + $10 / square foot
The maintenance cost is a flat fee of $100,000 per year, plus $10 multiplied by the size of the installation in square feet.
```

### Original quote/transcript 14

```text
Total cost: Land cost + Solar panel cost + Maintenance cost
```

### Original quote/transcript 15

```text
Let’s calculate the total cost using the actual solution:
```

### Original quote/transcript 16

```text
Total cost = (100 * x) + (250 * x) + (100,000 + (10 * x))
 = 100x + 250x + 100,000 + 10x
 = 360x + 100,000
```

### Original quote/transcript 17

```text
Is the student’s solution the same as the actual solution just calculated:
No
```

### Original quote/transcript 18

```text
Student grade:
Incorrect
```

### Original quote/transcript 19

```text
The AeroGlide UltraSlim Smart Toothbrush by Boie is a technologically advanced toothbrush designed to provide a superior brushing experience. Boie is a company known for its innovative oral care products, and the AeroGlide UltraSlim Smart Toothbrush is no exception.
```

### Original quote/transcript 20

```text
One of the standout features of this toothbrush is its ultra-slim design. The brush head is only 2mm thick, making it much thinner than traditional toothbrushes. This slim profile allows for better access to hard-to-reach areas of the mouth, ensuring a thorough and effective clean.
```

### Original quote/transcript 21

```text
The AeroGlide UltraSlim Smart Toothbrush also incorporates smart technology. It connects to a mobile app via Bluetooth, allowing users to track their brushing habits and receive personalized recommendations for improving their oral hygiene routine. The app provides real-time feedback on brushing technique, duration, and coverage, helping users to achieve optimal oral health.
```

### Original quote/transcript 22

```text
The toothbrush features soft, antimicrobial bristles made from a durable thermoplastic elastomer. These bristles are gentle on the gums and teeth, while also being effective at removing plaque and debris. The antimicrobial properties of the bristles help to inhibit the growth of bacteria, keeping the brush clean and hygienic.
```

### Original quote/transcript 23

```text
In terms of battery life, the AeroGlide UltraSlim Smart Toothbrush boasts an impressive 30-day battery life on a single charge. This makes it convenient for travel and ensures that users don’t have to worry about constantly recharging their toothbrush.
```

### Original quote/transcript 24

```text
Overall, the AeroGlide UltraSlim Smart Toothbrush by Boie offers a combination of advanced technology, slim design, and effective cleaning capabilities. It is a great option for those looking to upgrade their oral care routine and achieve a healthier smile.
```